In [28]:
import os
import random
import shutil
from pathlib import Path

# =========================
# CONFIG
# =========================
root_path = Path(r"D:\Project\DAP_paper\datasets\processed\DollarStreet")

train_dir = root_path / "train"
val_dir = root_path / "val"

seed = 42
val_ratio_from_train = 0.25   # because 25% of 80% = 20%
move_files = True             # True = move images, False = copy images

valid_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

random.seed(seed)

# =========================
# SPLIT TRAIN INTO VAL
# =========================
total_moved = 0

for region_dir in sorted(train_dir.iterdir()):
    if not region_dir.is_dir():
        continue

    region = region_dir.name

    for class_dir in sorted(region_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name

        images = [
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in valid_exts
        ]

        n = len(images)

        if n <= 1:
            print(f"Skip {region}/{class_name}: only {n} image")
            continue

        random.shuffle(images)

        val_count = round(n * val_ratio_from_train)

        # Make sure tiny classes do not fully disappear from train
        val_count = max(1, val_count)
        val_count = min(val_count, n - 1)

        val_class_dir = val_dir / region / class_name
        val_class_dir.mkdir(parents=True, exist_ok=True)

        selected_images = images[:val_count]

        for img_path in selected_images:
            dst_path = val_class_dir / img_path.name

            if move_files:
                shutil.move(str(img_path), str(dst_path))
            else:
                shutil.copy2(str(img_path), str(dst_path))

            total_moved += 1

        print(
            f"{region}/{class_name}: "
            f"total={n}, moved_to_val={val_count}, remaining_train={n - val_count}"
        )

print("=" * 50)
print(f"Done. Total images moved to val: {total_moved}")

af/409_analog clock: total=3, moved_to_val=1, remaining_train=2
af/412_trash can: total=64, moved_to_val=16, remaining_train=48
af/418_ballpoint: total=36, moved_to_val=9, remaining_train=27
af/428_wheelbarrow: total=8, moved_to_val=2, remaining_train=6
af/453_bookcase: total=45, moved_to_val=11, remaining_train=34
af/477_tool kit: total=43, moved_to_val=11, remaining_train=32
af/487_cellphone: total=39, moved_to_val=10, remaining_train=29
af/487_mobile phone: total=4, moved_to_val=1, remaining_train=3
af/527_desktop computer: total=17, moved_to_val=4, remaining_train=13
af/529_diaper: total=18, moved_to_val=4, remaining_train=14
af/531_digital watch: total=5, moved_to_val=1, remaining_train=4
af/532_dining table: total=80, moved_to_val=20, remaining_train=60
af/533_dishrag: total=102, moved_to_val=26, remaining_train=76
af/534_dishwasher: total=39, moved_to_val=10, remaining_train=29
af/545_electric fan: total=33, moved_to_val=8, remaining_train=25
af/567_skillet: total=67, moved_to_v

In [29]:
import pandas as pd
from pathlib import Path

root_path = Path(r"D:\Project\DAP_paper\datasets\processed\DollarStreet")
valid_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

rows = []

for split in ["train", "val", "test"]:
    split_dir = root_path / split

    if not split_dir.exists():
        continue

    for region_dir in sorted(split_dir.iterdir()):
        if not region_dir.is_dir():
            continue

        for class_dir in sorted(region_dir.iterdir()):
            if not class_dir.is_dir():
                continue

            count = sum(
                1 for p in class_dir.iterdir()
                if p.is_file() and p.suffix.lower() in valid_exts
            )

            rows.append({
                "split": split,
                "region": region_dir.name,
                "class": class_dir.name,
                "num_images": count
            })

df_counts = pd.DataFrame(rows)

display(df_counts)

summary = df_counts.groupby("split")["num_images"].sum().reset_index()
display(summary)

region_summary = df_counts.groupby(["split", "region"])["num_images"].sum().reset_index()
display(region_summary)

,split,region,class,num_images
0,train,af,409_analog clock,2
1,train,af,412_trash can,48
2,train,af,418_ballpoint,27
3,train,af,428_wheelbarrow,6
4,train,af,453_bookcase,34
...,...,...,...,...
758,test,eu,910_wooden spoon,6
759,test,eu,919_street sign,26
760,test,eu,923_plate,20
761,test,eu,968_cup,5


,split,num_images
0,test,4308
1,train,12919
2,val,4309


,split,region,num_images
0,test,af,847
1,test,am,856
2,test,as,1919
3,test,eu,686
4,train,af,2481
5,train,am,2538
6,train,as,5889
7,train,eu,2011
8,val,af,825
9,val,am,846
